### MCP Server dedicated to Cats ;)
My implementation of MCP server that serves:
 - random images of Cats with the option to select breed
 - information about breeds

Data fetched from `https://api.thecatapi.com/v1`

In [ ]:
from contextlib import AsyncExitStack
import html
import os
from pathlib import Path
from typing import Annotated

from agents import Agent, Runner, trace
from agents import Agent, Runner
from agents.mcp import MCPServerStdio
from agents.mcp import MCPServerStdio
from dotenv import find_dotenv, load_dotenv
import gradio as gr
from pydantic import BaseModel, Field

load_dotenv(find_dotenv())

API_KEY = os.getenv('CAT_API_KEY')

#### Models

In [ ]:
Score = Annotated[int, Field(ge=0, le=5)]

class CatScores(BaseModel):
    adaptability: Score = Field(description="The adaptability score of the breed")
    affection_level: Score = Field(description="The affection level score of the breed")
    child_friendly: Score = Field(description="The child friendly score of the breed")
    dog_friendly: Score = Field(description="The dog friendly score of the breed")
    energy_level: Score = Field(description="The energy level score of the breed")
    grooming: Score = Field(description="The grooming score of the breed")
    health_issues: Score = Field(description="The health issues score of the breed")
    intelligence: Score = Field(description="The intelligence score of the breed")
    shedding_level: Score = Field(description="The shedding level score of the breed")
    social_needs: Score = Field(description="The social needs score of the breed")
    stranger_friendly: Score = Field(description="The stranger friendly score of the breed")
    vocalisation: Score = Field(description="The vocalisation score of the breed")

class CatProperties(BaseModel):
    experimental: bool = Field(description="Whether the breed is experimental")
    hairless: bool = Field(description="Whether the breed is hairless")
    hypoallergenic: bool = Field(description="Whether the breed is hypoallergenic")
    indoor: bool = Field(description="Whether the breed is indoor")
    lap: bool = Field(description="Whether the breed is lap")
    natural: bool = Field(description="Whether the breed is natural")
    rare: bool = Field(description="Whether the breed is rare")
    rex: bool = Field(description="Whether the breed is rex")
    short_legs: bool = Field(description="Whether the breed is short legs")
    suppressed_tail: bool = Field(description="Whether the breed is suppressed tail")

class CatImage(BaseModel):
    url: str = Field(..., description="The URL of the image of a cat of the breed.", min_length=10)
    breed_name: str = Field(..., description="The name of the breed on the image.")

class BreedProperties(BaseModel):
    properties: CatProperties = Field(...,description="Properties of the breed")
    breed_name: str = Field(..., description="The name of the breed")

class BreedScores(BaseModel):
    scores: CatScores = Field(..., description="Scores of the breed")
    breed_name: str = Field(..., description="The name of the breed")

class ResponseFormat(BaseModel):
    description: str = Field(..., description="3-4 paragraphs summarizing cats of the breed", min_length=500)
    bp: list[BreedProperties] = Field(..., description="Properties of the breeds")
    bs: list[BreedScores] = Field(..., description="Scores of the breeds")
    image_urls: list[CatImage] = Field(..., description="The urls to the images of a cat of given breed.", min_length=2, max_length=5)


#### HTML-related stuff

In [ ]:
_PAGE_CSS = """
<style>
  .cat-muted { color: #64748b; font-size: 0.9rem; }
  .cat-chips { display: flex; flex-wrap: wrap; gap: 0.4rem; }
  .cat-chip {
    display: inline-block; padding: 0.25rem 0.65rem; border-radius: 999px;
    background: #f0fdfa; color: black; font-size: 0.82rem; font-weight: 500;
    border: 1px solid #99f6e4;
  }
  .cat-breed-props { margin-bottom: 1.25rem; }
  .cat-breed-props:last-child { margin-bottom: 0; }
  .cat-breed-title {
    margin: 0 0 0.5rem 0;
    font-size: 0.95rem;
    font-weight: 600;
    color: #0f766e;
  }
</style>
"""
_CSS = """
    <style>
      .cat-scores { font-family: ui-sans-serif, system-ui, sans-serif; max-width: 520px; }
      .cat-scores .row { display: grid; grid-template-columns: 9.5rem 1fr 2.5rem; align-items: center; gap: 0.5rem; margin: 0.35rem 0; }
      .cat-scores .lab { font-size: 0.82rem; color: #334155; }
      .cat-scores .track { height: 0.55rem; background: #e2e8f0; border-radius: 999px; overflow: hidden; }
      .cat-scores .fill { height: 100%; border-radius: 999px; background: linear-gradient(90deg, #0d9488, #14b8a6); transition: width 0.35s ease; }
      .cat-scores .num { font-size: 0.78rem; font-variant-numeric: tabular-nums; color: #64748b; text-align: right; }
    </style>
    """

_DEFAULT_LAYOUT = (
            "*Enter a question about cats above.*",
            _PAGE_CSS + "<p class='cat-muted'>Waiting for input.</p>",
            _PAGE_CSS + "",
            None,
        )



_SCORE_LABELS = {field: field.capitalize().replace("_", "-") for field in CatScores.model_fields}
_PROP_LABELS = {field: field.capitalize().replace("_", " ") for field in CatProperties.model_fields}

def _properties_html(props: list[BreedProperties]) -> str:
    if not props:
        return "<p class='cat-muted'>No breed properties in this response.</p>"
    blocks: list[str] = []
    for bp in props:
        title = html.escape(bp.breed_name.strip())
        active = [
            _PROP_LABELS.get(k, k.replace("_", " ").title())
            for k, v in bp.properties.model_dump().items()
            if v
        ]
        if active:
            chips = "".join(
                f"<span class='cat-chip'>{html.escape(label)}</span>" for label in active
            )
            inner = f"<div class='cat-chips'>{chips}</div>"
        else:
            inner = "<p class='cat-muted'>No boolean traits are <code>True</code> for this breed.</p>"
        blocks.append(
            f"<section class='cat-breed-props'><h4 class='cat-breed-title'>{title}</h4>{inner}</section>"
        )
    return "".join(blocks)


def _scores_html(scores: list[BreedScores]) -> str:
    if not scores:
        return "<p class='cat-muted'>No breed scores in this response.</p>"
    blocks: list[str] = []
    for bs in scores:
        title = html.escape(bs.breed_name.strip())
        rows: list[str] = []
        for key, val in bs.scores.model_dump().items():
            label = _SCORE_LABELS.get(key, key.replace("_", " ").title())
            pct = max(0, min(100, (val / 5) * 100))
            rows.append(
                f"<div class='row'><span class='lab'>{html.escape(label)}</span>"
                f"<div class='track'><div class='fill' style='width:{pct:.1f}%'></div></div>"
                f"<span class='num'>{int(val)}/5</span></div>"
            )
        inner = f"<div class='cat-scores'>{''.join(rows)}</div>"
        blocks.append(
            f"<section class='cat-breed-props'><h4 class='cat-breed-title'>{title}</h4>{inner}</section>"
        )
    return _CSS + "".join(blocks)


def _gallery_value(images: list[CatImage]) -> list[tuple[str, str]] | None:
    if not images:
        return None
    return [(img.url.strip(), img.breed_name.strip()) for img in images]

def get_error_layout(err: str) -> tuple[str, str, str, list[tuple[str, str]] | None]:
    return (
            f"**Error**\n\n```\n{err}\n```",
            _PAGE_CSS + f"<p class='cat-muted'>—</p>",
            _PAGE_CSS + "",
            None,
        )

#### Constants

In [ ]:
SERVER_NAME = Path('.') / 'cat_server.py'
MCP_SERVER_PARAMS = [
    {"command": "uv", "args": ["run", str(SERVER_NAME)]},
    {"command": "uvx",
        "args": ["--from", "git+https://github.com/garylab/serper-mcp-server.git", "serper-mcp-server"],
        "env": {"SERPER_API_KEY": os.getenv("SERPER_API_KEY")}},
]
INSTRUCTIONS = """You are Cat-Focused-Assistant serving the information about cats.
    Use your tools to gather diverse set of information about cats from the Uesr question. Use minimum 3 different sources of information.
    Create narrative desciption about the facts gatered and format that in markdown. Mention sources of the information.
    
    Fill in `bp` with properties of each breed.
    Fill in `bs` with scores of each breed.
    Fill in `image_urls` with urls of the images of each breed.
    
    DO NOT:
        - Embedd the image in the markdown.
        - Describe how did you get the information. 
    """

#### Main routine

In [ ]:
async def run_cat_assistant(user_request: str):
    if not user_request or not str(user_request).strip():
        return _DEFAULT_LAYOUT
    
    try:
        async with AsyncExitStack() as stack:
            mcp_servers = [
                await stack.enter_async_context(
                    MCPServerStdio(p, client_session_timeout_seconds=30)
                )
                for p in MCP_SERVER_PARAMS
            ]
            agent = Agent(
                model="gpt-5-nano",
                name="Cat Assistant",
                instructions=INSTRUCTIONS,
                mcp_servers=mcp_servers,
                output_type=ResponseFormat,
            )
            with trace('cat-agent'):
                result = await Runner.run(agent, user_request.strip())
            rf: ResponseFormat = result.final_output

        desc = rf.description
        props = _PAGE_CSS + _properties_html(rf.bp)
        scores = _PAGE_CSS + _scores_html(rf.bs)
        gallery = _gallery_value(rf.image_urls)
        return desc, props, scores, gallery
    except Exception as e:
        err = html.escape(str(e))
        return get_error_layout(err)


with gr.Blocks(title="Cat Assistant") as cat_demo:
    gr.Markdown("## Cat assistant")
    req = gr.Textbox(
        label="Your request",
        placeholder="e.g. Tell me about cats from Turkey - pick at least two breeds.",
        lines=3,
    )
    go = gr.Button("Run", variant="primary")
    gr.Markdown("### Image")
    out_img = gr.Gallery(
        label="Images",
        columns=3,
        object_fit="contain",
        allow_preview=True,
        show_label=False,
        height=350,
        buttons=['fullscreen', 'download']
    )
    gr.Markdown("### Narrative")
    out_desc = gr.Markdown()
    gr.Markdown("### Active traits")
    out_props = gr.HTML()
    gr.Markdown("### Scores (0-5)")
    out_scores = gr.HTML()
    

    go.click(
        fn=run_cat_assistant,
        inputs=[req],
        outputs=[out_desc, out_props, out_scores, out_img],
    )

cat_demo.launch(inbrowser=True, share=False, inline=False, theme=gr.themes.Soft(primary_hue="teal", neutral_hue="slate"),)

In [ ]:
gr.close_all()